# Student Identification & Github Repository URL

Please complete the following information before submitting your notebook.

**UNI:xy2703, zy2744, jz3998**  
**Full Name: Esme You, Ziyan Yu, Joey Zhu** (alphabetical order)

**Public GitHub Repository URL (Final Project Report & Code):** https://github.com/Exsy777/QMSSGR5074_Project_1.git


# World Happiness Classification Competition

## Project Objectives

The objective of this project is to build and evaluate machine learning models that classify countries into happiness categories based on economic, social, and development indicators. Through this process, the project aims to deepen understanding of how machine learning models learn patterns from data and how model design choices influence predictive performance.

---

## Project Workflow & Outline

1. **Load and Merge Datasets**  
   - Import all relevant datasets  
   - Perform necessary joins and validation  

2. **Data Preprocessing**  
   - Build preprocessing pipelines using `sklearn`'s `ColumnTransformer`  
   - Handle missing values, encoding, scaling, and feature engineering  

3. **Model Training**  
   - Fit the model using the processed data  
   - Save both the preprocessing pipeline and trained model  

4. **Prediction and Evaluation**  
   - Generate predictions  
   - Evaluate performance using appropriate metrics  

5. **Hyperparameter Tuning**  
   - Systematically tune model parameters  
   - Compare performance across configurations  

6. **Deep Learning Experimentation**  
   - Implement and evaluate a neural network approach  
   - Compare results with classical ML models  

7. **Model Explainability (SHAP)**  
   - Apply SHAP for feature importance analysis  
   - Interpret and explain model decisions  


---

## 0. Loading Datasets

In this section, the World Happiness 2023 datasets are loaded into the notebook environment to serve as the primary data sources for the analysis. The datasets are inspected to understand their structure, including the number of observations, available variables, column names, and data types. This initial inspection helps verify that the datasets were imported correctly and ensures that key variables required for later analysis are present.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Import drive to Google Colab
from google.colab import drive
drive.mount('/content/drive')

# Load the dataset
whr_df = pd.read_csv('/content/drive/MyDrive/Projects in ML/WHR_2023.csv')

# Inspect the first few rows to understand the structure
whr_df


In [ ]:
# Convert the regression target ('happiness_score') into classification labels

# Define quartiles
whr_df['happiness_category'] = pd.qcut(whr_df['happiness_score'],
                                       q=5,
                                       # 4 happiness categories: Very Low, Low, High, Very High
                                       labels=['Very Low', 'Low','Average', 'High', 'Very High'])

# Select features and target
X = whr_df.drop(columns=['happiness_score', 'happiness_category'])
y = whr_df['happiness_category']

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Convert y_train and y_test to numerical labels
y_train_labels = y_train.astype('category').cat.codes
# y_test_labels = ## Complete in a similar manner as above. TODO
y_test_labels = y_test.astype('category').cat.codes

### Target Variable Encoding and Label Representation

The target variable is converted into a categorical format and encoded into numerical codes so it can be used by machine learning models that require numeric inputs. The encoded version (y_train_labels) provides integer representations of the categories, while the original variable (y_train) retains the human-readable labels used for interpretation and evaluation.






1. `y_train.astype('category').cat.codes` converts the categorical target variable into numerical labels that machine learning models can use. The astype('category') step ensures the variable is treated as a categorical data type, and .cat.codes assigns an integer value to each category (for example, Very Low = 0, Low = 1, Average = 2, High = 3, Very High = 4).
2. The difference between y_train_labels and y_train is that y_train contains the original categorical class labels, while y_train_labels contains their numeric representations. Both variables describe the same target information, but the numeric version is required for model training and computation, whereas the original categorical labels are easier to interpret when analyzing results.

### Add New Data

In [ ]:
# Truncated and cleaned up region data to merge
countrydata=pd.read_csv('/content/drive/MyDrive/Projects in ML/newcountryvars.csv')

countrydata.head()

In [ ]:
# Organize datasets
whr_countries = set(X_train['country'].dropna().unique())
new_countries = set(countrydata['country_name'].dropna().unique())

common_countries = whr_countries.intersection(new_countries)
only_in_whr = whr_countries - new_countries
only_in_new = new_countries - whr_countries

print(f"Countries in WHR (train): {len(whr_countries)}")
print(f"Countries in newcountryvars: {len(new_countries)}")
print(f"Common countries: {len(common_countries)}")
print(f"Only in WHR: {len(only_in_whr)}")
print(f"Only in newcountryvars: {len(only_in_new)}")

# Merge
X_train = X_train.merge(countrydata, left_on='country', right_on='country_name', how='left')
X_test  = X_test.merge(countrydata, left_on='country', right_on='country_name', how='left')


# Drop duplicate columns created by prior merges (_x/_y)
X_train = X_train.loc[:, ~X_train.columns.str.endswith(('_x', '_y'))]
X_test  = X_test.loc[:, ~X_test.columns.str.endswith(('_x', '_y'))]


In [ ]:
X_train.head(1)

---

## 1. Exploratory Data Analysis and Visualization (EDAV)

In [ ]:
print(X_train.dtypes)

### Observations

The dataset contains both categorical and numerical variables. The categorical variables include country and region, while the remaining features are continuous socioeconomic indicators such as GDP per capita, social support, life expectancy, HDI, GNI, and education-related variables. Most numerical variables are stored as float64, which is appropriate for modeling. After merging the additional country-level dataset, several development indicators (e.g., population, HDI, and GNI) were successfully appended. The presence of both categorical and numerical data types indicates that separate preprocessing strategies will be required. Categorical variables will need encoding, and numerical variables may require scaling and imputation. The dataset appears structurally consistent after removing duplicate merge columns, and it is suitable for further exploratory analysis and modeling.

### Missing Values Analysis

In this section, the number and percentage of missing values are calculated for each variable to identify potential data gaps and inform appropriate imputation strategies before modeling.

In [ ]:

# Compute missing counts and percentages
missing_summary = pd.DataFrame({
    "Missing Count": X_train.isnull().sum(),
    "Missing Percentage (%)": (X_train.isnull().sum() / len(X_train)) * 100
})
# Sort by highest missing percentage
missing_summary = missing_summary.sort_values(
    by="Missing Percentage (%)",
    ascending=False
)

missing_summary

### Distribution of Key Numerical Features

In this section, histograms are used to visualize the distributions of selected numerical variables that may influence the target variable. Examining these distributions helps identify patterns such as skewness, concentration of values, and potential outliers that may affect model performance or require transformation.

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

# Select relevant numerical features likely to influence happiness
num_features = [
    'gdp_per_capita',
    'social_support',
    'healthy_life_expectancy',
    'freedom_to_make_life_choices',
    'hdi',
    'gni'
]

# Remove any features not present (safe check)
num_features = [col for col in num_features if col in X_train.columns]

plt.figure(figsize=(14, 10))

for i, col in enumerate(num_features):
    plt.subplot(3, 2, i+1)
    sns.histplot(X_train[col], kde=True, bins=20)
    plt.title(f'Distribution of {col}')

plt.tight_layout()
plt.show()

### Distribution of Categorical Variables

In this section, the distributions of relevant categorical variables are visualized using bar charts to examine how observations are distributed across categories. This helps identify potential class imbalances or patterns that may influence model training and interpretation.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create cross-tab
region_happiness = pd.crosstab(X_train['region'], y_train)

# Plot stacked bar chart
region_happiness.plot(kind='bar', stacked=True, figsize=(10,6))

plt.title("Happiness Category Distribution by Region")
plt.xlabel("Region")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.legend(title="Happiness Category")
plt.tight_layout()
plt.show()

# Proportion Plot
region_prop = pd.crosstab(X_train['region'], y_train, normalize='index')

region_prop.plot(kind='bar', stacked=True, figsize=(10,6))

plt.title("Proportion of Happiness Categories by Region")
plt.xlabel("Region")
plt.ylabel("Proportion")
plt.xticks(rotation=45)
plt.legend(title="Happiness Category")
plt.tight_layout()
plt.show()

The results indicate systematic regional clustering of happiness levels. Developed regions tend to concentrate in higher happiness categories, while lower-income regions exhibit greater representation in lower categories. This geographic structure supports the inclusion of region as a meaningful explanatory variable in the modeling pipeline.

### Feature Correlation Analysis

In this section, correlations among numerical features are examined using Pearson, Spearman, and Kendall correlation coefficients. Correlation matrices and heatmaps are used to visualize the strength and direction of relationships between variables. This analysis helps identify strong positive or negative associations and detect potential multicollinearity among development indicators, which may influence feature selection and modeling strategies.

In [ ]:

# Select only numerical columns
numeric_cols = X_train.select_dtypes(include=['float64', 'int64'])

# Compute correlation matrices
pearson_corr = numeric_cols.corr(method='pearson')
spearman_corr = numeric_cols.corr(method='spearman')
kendall_corr = numeric_cols.corr(method='kendall')

# Plot Pearson
plt.figure(figsize=(10,8))
sns.heatmap(pearson_corr, cmap='coolwarm', center=0)
plt.title("Pearson Correlation Matrix")
plt.show()

# Plot Spearman
plt.figure(figsize=(10,8))
sns.heatmap(spearman_corr, cmap='coolwarm', center=0)
plt.title("Spearman Correlation Matrix")
plt.show()

# Plot Kendall
plt.figure(figsize=(10,8))
sns.heatmap(kendall_corr, cmap='coolwarm', center=0)
plt.title("Kendall Correlation Matrix")
plt.show()

The correlation analysis shows strong positive relationships among key development indicators such as GDP per capita, GNI, HDI, life expectancy, and education variables. These variables are highly correlated across Pearson, Spearman, and Kendall measures, indicating consistent linear and monotonic associations. Poverty rate exhibits strong negative correlations with income, education, and health indicators. The high correlation between GDP and GNI, as well as between HDI and its components, suggests potential multicollinearity. While this is less concerning for tree-based models, it may affect interpretability in linear models.

### Bivariate Analysis and Relationship Exploration

In this section, relationships between features and the target variable are examined using bivariate visualizations and summary statistics. Scatter plots, box plots, and grouped comparisons help illustrate how feature distributions vary across happiness categories, while correlation tables highlight associations between variables. This analysis identifies meaningful patterns and relationships that may influence model performance and feature selection.

In [ ]:

# Combine features and target for plotting
train_df = X_train.copy()
train_df["happiness_category"] = y_train.values

# Boxplots of key numerical variables vs target
key_features = [
    "gdp_per_capita",
    "social_support",
    "healthy_life_expectancy",
    "hdi",
    "gni"
]

key_features = [col for col in key_features if col in train_df.columns]

plt.figure(figsize=(15,10))
for i, col in enumerate(key_features):
    plt.subplot(3, 2, i+1)
    sns.boxplot(x="happiness_category", y=col, data=train_df)
    plt.xticks(rotation=45)
    plt.title(f"{col} by Happiness Category")
plt.tight_layout()
plt.show()


# Compare mean values of key features across happiness categories
grouped_means = train_df.groupby("happiness_category")[key_features].mean()

grouped_means

The bivariate analysis reveals a strong and consistent upward trend in development indicators across happiness categories. GDP per capita, GNI, HDI, life expectancy, and social support increase progressively from “Very Low” to “Very High” groups, indicating a clear association between higher socioeconomic development and greater happiness classification. The stepwise increase in group means suggests a structured gradient rather than random variation. These patterns imply that economic capacity and human development metrics are likely to play a central role in model prediction and may provide substantial discriminatory power between happiness levels.

### Outlier Detection

In this section, potential outliers in numerical features are identified using methods such as box plots, Z-score analysis, and the Interquartile Range (IQR) approach. Detecting extreme values helps determine whether observations represent genuine variation or anomalies that may require transformation or special handling during preprocessing.

In [ ]:

from scipy import stats

# Z-score method
z_scores = np.abs(stats.zscore(numeric_cols, nan_policy='omit'))
z_outliers = (z_scores > 3).sum(axis=0)

z_outlier_summary = pd.Series(z_outliers, index=numeric_cols.columns)
z_outlier_summary.sort_values(ascending=False)


# IQR method
Q1 = numeric_cols.quantile(0.25)
Q3 = numeric_cols.quantile(0.75)
IQR = Q3 - Q1

iqr_outliers = ((numeric_cols < (Q1 - 1.5 * IQR)) |
                (numeric_cols > (Q3 + 1.5 * IQR))).sum()

iqr_outliers.sort_values(ascending=False)

Outlier detection reveals that perceptions of corruption and population contain the largest number of extreme values. Population outliers are expected due to the substantial scale differences between small and highly populous countries. Similarly, perceptions of corruption may exhibit extreme values reflecting significant institutional differences across nations. Income-related variables such as GDP and GNI show limited but present high-end outliers, likely corresponding to very high-income countries. Development indicators such as HDI, life expectancy, and schooling variables do not exhibit substantial outlier behavior.

Because these values represent genuine cross-country heterogeneity rather than data errors, they should be retained. However, log transformation of highly skewed variables such as GDP, GNI, and population may help stabilize variance and reduce the influence of extreme observations. Tree-based models are generally robust to outliers, so removal is not recommended.

### Observations and General Comments



Economic indicators such as GDP per capita, GNI, HDI, life expectancy, and education variables increase consistently across happiness levels, indicating strong predictive relevance. Regional patterns suggest geographic clustering of development and happiness outcomes. Missing values are primarily introduced through the merged country-level dataset and will require imputation rather than row removal to preserve sample size. Outlier detection shows extreme values mainly in population and high-income variables; these reflect genuine cross-country variation and should be retained, though log transformation of scale-heavy variables (e.g., GDP, GNI, population) may improve stability. Prior to modeling, categorical variables must be encoded, numerical features scaled, and missing values imputed to ensure a robust preprocessing pipeline.

---

## 2. Feature Engineering

In this section, log transformations are applied to highly skewed numerical features to reduce the influence of extreme values and stabilize variance. Normalizing skewed variables helps improve model stability and can make relationships between variables easier for machine learning models to learn.

In [ ]:

# Identify skewed, scale-heavy variables
log_features = ['gdp_per_capita', 'gni', 'population']

# Apply log(1 + x) transformation safely
for col in log_features:
    if col in X_train.columns:
        X_train[col] = np.log1p(X_train[col])
        X_test[col] = np.log1p(X_test[col])

# Optional: verify transformation
X_train[log_features].head()

Interaction features are created to capture combined effects between related variables that may influence the target outcome. By modeling interactions between features such as economic indicators and social factors, the model can better represent nonlinear relationships and potentially improve predictive performance.

In [ ]:
# Create interaction features
# Economic capacity × social support
if all(col in X_train.columns for col in ['gdp_per_capita', 'social_support']):
    X_train['gdp_social_interaction'] = X_train['gdp_per_capita'] * X_train['social_support']
    X_test['gdp_social_interaction'] = X_test['gdp_per_capita'] * X_test['social_support']

# Human development × freedom
if all(col in X_train.columns for col in ['hdi', 'freedom_to_make_life_choices']):
    X_train['hdi_freedom_interaction'] = X_train['hdi'] * X_train['freedom_to_make_life_choices']
    X_test['hdi_freedom_interaction'] = X_test['hdi'] * X_test['freedom_to_make_life_choices']

# Verify
X_train[['gdp_social_interaction', 'hdi_freedom_interaction']].head()

---

## 3. Data Preprocessing

In this section, a preprocessing pipeline is constructed using sklearn’s ColumnTransformer to systematically prepare the data for modeling. The pipeline handles numerical and categorical features separately, applying appropriate transformations such as imputation, scaling, and encoding. The fitted preprocessor is saved so that the same transformations can be consistently applied to training data, test data, and future predictions.

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Create the preprocessing pipelines for both numeric and categorical data.

numeric_features = X_train.select_dtypes(include=['int64', 'float64'])
numeric_features=numeric_features.columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

categorical_features = ['region'] # Changed from ['region', 'sub-region']

#Replacing missing values with Modal value and then one hot encoding.
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

# final preprocessor object set up with ColumnTransformer
preprocessor = ColumnTransformer(transformers=[('num', numeric_transformer, numeric_features),('cat', categorical_transformer, categorical_features)])

#Fit preprocessor object
preprocess = preprocessor.fit(X_train)

### Explanation



The preprocessing pipeline applies separate transformations to numerical and categorical features. Numerical features are imputed using the median and then standardized with StandardScaler. Median imputation was selected because many economic variables are skewed, and the median is more robust to extreme values than the mean or a constant value. Standardization ensures that variables are placed on comparable scales before modeling.

Categorical variables are imputed using the most frequent category and encoded with one-hot encoding. This converts categorical features into binary indicators without imposing an ordinal structure. The handle_unknown='ignore' setting prevents errors if unseen categories appear in the test set.These steps ensure the data is complete, scaled appropriately, and formatted correctly for model training.



In [ ]:
# function to transform data with preprocessor

def preprocessor(data):
    data.drop(['country', 'region'], axis=1)
    preprocessed_data=preprocess.transform(data)
    return preprocessed_data

### Preprocessing Pipeline Components and Their Roles

The preprocessor object is the ColumnTransformer definition that specifies how numerical and categorical features should be transformed. It contains the transformation logic (imputation, scaling, encoding) but is not yet fitted to the data.

The preprocess object is the fitted version of the preprocessor. After calling .fit(X_train), it stores the learned parameters from the training data, such as medians for imputation and scaling statistics (mean and standard deviation). This fitted object is what should be used to transform new data.

The preprocessor function (e.g., preprocess_data(data)) is a wrapper function that applies the fitted preprocess object to any dataset. It provides a convenient way to consistently transform training, validation, or test data using the same learned parameters.

The final preprocessed_data returned is the transformed numerical matrix produced by applying the fitted preprocessing pipeline to a dataset. It contains scaled numerical features and one-hot encoded categorical features in a format ready for model training or prediction.

In [ ]:
# check shape of X data after preprocessing it using our new function
preprocessor(X_train).shape

---

## 4. Model Training and Saving Artifacts
In this section, the machine learning model is trained using the preprocessed training data. After fitting the model, both the trained model and the fitted preprocessing pipeline are saved so they can be reused for future predictions, evaluation, or deployment.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib

# Transform training data using fitted preprocessor
X_train_processed = preprocess.transform(X_train)

# Define + fit Random Forest (baseline)
model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

model.fit(X_train_processed, y_train)

# Score on training set (0–1)
train_pred = model.predict(X_train_processed)
train_score = accuracy_score(y_train, train_pred)
train_score

---

## 5. Model Evaluation

In this section, the trained model generates predictions for the test dataset (X_test). These predictions are compared with the true labels (y_test) using evaluation metrics such as accuracy, precision, recall, and F1-score. This evaluation assesses how well the model generalizes to unseen data and provides insight into its classification performance across different target categories.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Transform test data
X_test_processed = preprocess.transform(X_test)

# Generate predictions
prediction_labels = model.predict(X_test_processed)

# Accuracy
print("Test Accuracy:", accuracy_score(y_test, prediction_labels))

# Classification report
print("\nClassification Report:\n")
print(classification_report(y_test, prediction_labels))

# Confusion matrix
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, prediction_labels))

The model achieves a test accuracy of 0.48, indicating moderate predictive performance above random guessing (0.20 for five balanced classes). Performance varies across categories. “Very High” shows strong precision and recall (F1 = 0.75), suggesting that extreme high-development countries are easier to classify. In contrast, “Low” and “Very Low” exhibit weaker precision and recall, with substantial confusion between adjacent categories. The confusion matrix shows that most misclassifications occur between neighboring happiness levels (e.g., “Low” vs “Very Low,” or “Average” vs “High”), indicating that the model struggles to distinguish borderline cases. Overall, the results suggest the model captures strong structural signals but may require hyperparameter tuning or additional feature engineering to improve discrimination among middle categories.

---

## 6. Hyperparameter Experimentation

The model is trained multiple times using different hyperparameter settings to evaluate how parameter choices affect predictive performance. By systematically adjusting parameters such as tree depth, number of estimators, or sample constraints, the experiments help identify configurations that improve generalization on the test data. Results from each experiment are compared, and the best-performing parameter combination is reported.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Transform train/test once
X_train_processed = preprocess.transform(X_train)
X_test_processed = preprocess.transform(X_test)

# Model 2: regularized Random Forest
model_2 = RandomForestClassifier(
    n_estimators=600,          # more trees for stability
    max_depth=8,               # limit depth to reduce overfitting
    min_samples_split=5,       # require more samples to split
    min_samples_leaf=2,        # require more samples per leaf
    max_features="sqrt",       # standard RF setting for classification
    random_state=42
)

# Fit
model_2.fit(X_train_processed, y_train)

# Evaluate
pred_2 = model_2.predict(X_test_processed)

print("Model 2 Test Accuracy:", accuracy_score(y_test, pred_2))
print("\nClassification Report (Model 2):\n")
print(classification_report(y_test, pred_2))

### Reflection on Hyperparameter Changes


In the second model, several hyperparameters were adjusted to reduce overfitting and improve generalization. The number of trees (n_estimators) was increased to improve stability and reduce variance. The maximum tree depth (max_depth) was limited to prevent the model from learning overly complex patterns specific to the training data. The parameters min_samples_split and min_samples_leaf were increased to require more observations before creating splits and leaf nodes, which further regularizes the model and reduces sensitivity to noise. The max_features="sqrt" setting limits the number of features considered at each split, encouraging diversity among trees.

These changes improved test accuracy from 0.48 to 0.52 and increased macro F1 from 0.48 to 0.53, indicating better balanced performance across categories. The baseline model achieved perfect training accuracy (1.0), suggesting severe overfitting. After regularization, the model generalizes better, particularly in middle categories such as “Average” and “Low.” The results demonstrate that controlling model complexity improves out-of-sample performance even if training accuracy decreases.

In [ ]:
#Evaluate Model 2:

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Transform test data
X_test_processed = preprocess.transform(X_test)

# Generate predictions (Model 2)
prediction_labels = model_2.predict(X_test_processed)

# Accuracy
print("Model 2 Test Accuracy:", accuracy_score(y_test, prediction_labels))

# Classification report
print("\nClassification Report (Model 2):\n")
print(classification_report(y_test, prediction_labels))

# Confusion matrix
print("\nConfusion Matrix (Model 2):\n")
print(confusion_matrix(y_test, prediction_labels))


### Hyperparameter Tuning Strategy and Optimization Methods

Further manual adjustment of hyperparameters may yield incremental improvements, but repeatedly testing random combinations is inefficient and unsystematic. While the regularized model improved accuracy from 0.48 to 0.52, performance gains are modest, suggesting that manual tuning may have diminishing returns. Instead of trial-and-error experimentation, a structured approach such as GridSearchCV or RandomizedSearchCV should be used. These methods systematically explore combinations of hyperparameters using cross-validation and select the configuration that maximizes performance on validation folds. This approach is more reliable, reproducible, and statistically sound than manually testing arbitrary parameter values.

In [ ]:
# Submit a third model using GridSearchCV

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

X_train_processed = preprocess.transform(X_train)
X_test_processed = preprocess.transform(X_test)

# Smaller grid (fast)
param_grid = {
    "n_estimators": np.arange(200, 601, 200),   # 200, 400, 600
    "max_depth": np.arange(6, 13, 2),           # 6, 8, 10, 12
    "min_samples_leaf": np.arange(1, 4, 1),     # 1, 2, 3
    "max_features": ["sqrt"]
}

rf = RandomForestClassifier(random_state=42)

gridmodel = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=1            # avoids Colab parallel overhead
)

gridmodel.fit(X_train_processed, y_train)

print("best mean cross-validation score: {:.3f}".format(gridmodel.best_score_))
print("best parameters: {}".format(gridmodel.best_params_))

best_model = gridmodel.best_estimator_
prediction_labels = best_model.predict(X_test_processed)

print("\nTest Accuracy (Best Grid Model):", accuracy_score(y_test, prediction_labels))
print("\nClassification Report:\n", classification_report(y_test, prediction_labels))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, prediction_labels))

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Transform test data (if not already done)
X_test_processed = preprocess.transform(X_test)

# Generate predictions using best grid model
prediction_labels = best_model.predict(X_test_processed)

# Accuracy
print("Model 3 (GridSearch) Test Accuracy:", accuracy_score(y_test, prediction_labels))

# Classification report
print("\nClassification Report (Model 3):\n")
print(classification_report(y_test, prediction_labels))

# Confusion matrix
print("\nConfusion Matrix (Model 3):\n")
print(confusion_matrix(y_test, prediction_labels))


In [ ]:
# Gradient Boosting
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import GradientBoostingClassifier


# Transform data
X_train_processed = preprocess.transform(X_train)
X_test_processed = preprocess.transform(X_test)

# Define alternative model (Gradient Boosting)
model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

# Fit model
model.fit(X_train_processed, y_train)

# Generate predictions
prediction_labels = model.predict(X_test_processed)

# Evaluate
print("Gradient Boosting Test Accuracy:", accuracy_score(y_test, prediction_labels))

print("\nClassification Report:\n")
print(classification_report(y_test, prediction_labels))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, prediction_labels))


### Model Parameter Configuration and Performance Analysis

In the regularized Random Forest model, the parameters n_estimators=600, max_depth=8, min_samples_split=5, min_samples_leaf=2, and max_features="sqrt" were defined to control model complexity and reduce overfitting. Increasing the number of trees improved stability, while limiting tree depth and requiring more samples per split and leaf reduced variance and prevented the model from memorizing the training data. This adjustment improved test accuracy from 0.48 in the baseline model (which had 1.0 training accuracy and clear overfitting) to 0.52, with macro F1 increasing from 0.48 to 0.53. The improvement in recall for middle categories such as “Average” and “Low” indicates better generalization.

In the Gradient Boosting model, n_estimators=200, learning_rate=0.05, and max_depth=3 were specified. The learning rate controls how strongly each tree corrects previous errors, and a smaller value promotes gradual, more stable learning. Shallow trees (max_depth=3) reduce overfitting while allowing the model to capture nonlinear relationships. Tree-based ensemble methods worked best overall because the dataset contains nonlinear interactions, multicollinearity among development indicators, and genuine outliers—conditions under which ensemble trees are robust and effective.

---

## 7. Basic Deep Learning

In [ ]:
# Now experiment with deep learning models:
import keras
from keras.models import Sequential
from keras.layers import Dense, Activation

# Preprocess input features
X_train_nn = preprocess.transform(X_train)
X_test_nn  = preprocess.transform(X_test)

if hasattr(X_train_nn, "toarray"):
    X_train_nn = X_train_nn.toarray()
    X_test_nn  = X_test_nn.toarray()

# Count features in input data
feature_count = X_train_nn.shape[1]

# Define Neural Network (5 layers: 128->64->64->32->output)
# Output neurons = number of classes (=5)

num_classes = y_train.nunique()

keras_model = Sequential([
    Dense(128, activation='relu', input_shape=(feature_count,)),
    Dense(64, activation='relu'),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])


# Encode categorical labels (one-hot)

y_train_cat = y_train.astype('category')
y_test_cat = y_test.astype('category')

# Ensure same category order in train/test
y_test_cat = y_test_cat.cat.set_categories(y_train_cat.cat.categories)

y_train_onehot = pd.get_dummies(y_train_cat).values
y_test_onehot  = pd.get_dummies(y_test_cat).values

# Compile model
keras_model.compile(loss='categorical_crossentropy', optimizer='sgd', metrics=['accuracy'])

# Fit model
history = keras_model.fit(
    X_train_nn, y_train_onehot,
    batch_size=20,
    epochs=300,
    validation_split=0.25,
    verbose=1
)


### Activation Functions and Output Layer Design

In the hidden layers, the ReLU (Rectified Linear Unit) activation function was used. ReLU was chosen because it introduces nonlinearity while remaining computationally efficient and helps prevent the vanishing gradient problem that can occur with sigmoid or tanh activations. Given the moderate depth of the network (128→64→64→32), ReLU supports stable gradient flow during training and allows the model to learn nonlinear relationships between socioeconomic indicators and happiness categories.

Softmax was used in the final layer because the task is multi-class classification with five mutually exclusive happiness categories. The final layer therefore contains five neurons, one for each class. Softmax converts the output logits into a probability distribution that sums to 1, enabling the model to assign class probabilities and select the category with the highest predicted likelihood. This activation is appropriate when using categorical cross-entropy loss and aligns with the structure of the one-hot encoded target labels.

### Training Duration and Epoch Selection (300)

Training for 300 epochs is not necessarily optimal and depends on validation performance. In small tabular datasets such as this one, long training durations increase the risk of overfitting. If validation accuracy plateaus early or begins to decline while training accuracy continues to increase, this indicates that the model is memorizing the training data rather than improving generalization.

Given the relatively small sample size and moderate model complexity, it is likely that the network converges well before 300 epochs. Training longer beyond the point of convergence does not improve test performance and may worsen it due to overfitting. A better approach would be to monitor validation loss and implement early stopping, which automatically halts training when performance stops improving. This ensures efficient training while preserving generalization.

Therefore, 300 epochs may be acceptable as an upper bound, but the decision should be guided by validation metrics rather than a fixed number of epochs.

### Loss Function and Optimization Strategy

loss='categorical_crossentropy' was used because this is a multi-class classification problem with mutually exclusive categories and one-hot encoded labels. Categorical cross-entropy measures the divergence between the predicted probability distribution (produced by the softmax output layer) and the true class distribution. Since the final layer uses softmax and the labels are one-hot encoded, categorical cross-entropy is the mathematically appropriate loss function for this setting.

optimizer='sgd' (Stochastic Gradient Descent) was used because it is a standard gradient-based optimization method that updates model weights iteratively using mini-batches. SGD is simple, stable, and often effective when learning rates are properly tuned. However, plain SGD can converge slowly and may require careful learning rate selection to achieve strong performance.

Given the moderate dataset size and network depth, switching to an adaptive optimizer such as Adam could improve convergence speed and stability. Adam automatically adjusts learning rates for each parameter, often reaching good solutions faster and with less manual tuning. Therefore, while categorical cross-entropy is appropriate and should not be changed, replacing SGD with Adam would likely improve training efficiency and potentially final validation performance.

### Plot Training Curves

In this section, the model’s training history is used to plot learning curves showing training and validation loss and accuracy across epochs. These curves help evaluate how the model learns over time and reveal potential issues such as overfitting or underfitting.

In [ ]:
## Your code to plot training and validation curves in a single plot (Make changes in the model cell to be able to do this) : TODO

# Single-plot learning curves
import matplotlib.pyplot as plt

hist = history.history
epochs = range(1, len(hist["loss"]) + 1)

fig, ax1 = plt.subplots(figsize=(10,6))

# Loss on left axis
ax1.plot(epochs, hist["loss"], label="Train Loss")
ax1.plot(epochs, hist["val_loss"], label="Val Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")

# Accuracy on right axis
ax2 = ax1.twinx()
ax2.plot(epochs, hist["accuracy"], label="Train Accuracy")
ax2.plot(epochs, hist["val_accuracy"], label="Val Accuracy")
ax2.set_ylabel("Accuracy")

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right")

plt.title("Training and Validation Curves (Loss and Accuracy)")
plt.tight_layout()
plt.show()

In [ ]:
#-- Generate predicted y values

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

# Generate predicted column index values
prediction_column_index = np.argmax(keras_model.predict(X_test_nn), axis=1)

# Extract correct prediction labels
label_names = y_train.astype('category').cat.categories
prediction_labels = [label_names[i] for i in prediction_column_index]

# Show model performance
print("Neural Network Test Accuracy:", accuracy_score(y_test, prediction_labels))

print("\nClassification Report:\n")
print(classification_report(y_test, prediction_labels))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, prediction_labels))


### Regularization: Dropout and Batch Normalization

Regularization techniques such as Dropout and Batch Normalization are applied to the neural network to improve generalization and reduce overfitting. Model performance is compared before and after applying these techniques by examining changes in training and validation loss and accuracy.

In [ ]:


# Difine Regularized Neural Network
from keras.models import Sequential
from keras.layers import Dense, Dropout, BatchNormalization
from keras.optimizers import Adam

# Build regularized model
keras_model_reg = Sequential([
    Dense(128, activation='relu', input_shape=(feature_count,)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),

    Dense(32, activation='relu'),
    BatchNormalization(),

    Dense(num_classes, activation='softmax')
])

keras_model_reg.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

history_reg = keras_model_reg.fit(
    X_train_nn, y_train_onehot,
    batch_size=20,
    epochs=300,
    validation_split=0.25,
    verbose=1
)

# Evaluate Regularized Model
test_loss_reg, test_acc_reg = keras_model_reg.evaluate(X_test_nn, y_test_onehot, verbose=0)
print("Regularized NN Test Accuracy:", test_acc_reg)

# Plot Before vs. After Regularization
import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))

# Accuracy comparison
plt.subplot(1,2,1)
plt.plot(history.history['val_accuracy'], label='Val Acc (No Reg)')
plt.plot(history_reg.history['val_accuracy'], label='Val Acc (Reg)')
plt.title("Validation Accuracy Comparison")
plt.legend()

# Loss comparison
plt.subplot(1,2,2)
plt.plot(history.history['val_loss'], label='Val Loss (No Reg)')
plt.plot(history_reg.history['val_loss'], label='Val Loss (Reg)')
plt.title("Validation Loss Comparison")
plt.legend()

plt.tight_layout()
plt.show()

After adding Dropout and Batch Normalization, validation accuracy became lower and more unstable compared to the unregularized model. The unregularized model reached validation accuracy around 0.45–0.50, while the regularized model remained closer to 0.33–0.42. More importantly, validation loss for the regularized model increases steadily after early epochs and grows much faster than the non-regularized model, indicating poorer convergence. The regularized network appears to underfit rather than improve generalization.

Given the relatively small dataset size, the original model may not have been severely overfitting, and adding strong regularization (Dropout 0.3) likely reduced the model’s ability to learn meaningful structure. In this case, regularization degraded performance rather than improving it. A better approach would be early stopping instead of heavy dropout. This demonstrates that regularization must be applied proportionally to dataset size and model complexity rather than assumed to always improve results.

### Activation Function Experimentation

Different activation functions—ReLU, LeakyReLU, Tanh, and Sigmoid—are tested to evaluate how they influence the neural network’s learning behavior and predictive performance. Comparing results across activation functions helps determine which function provides the most stable training and highest model accuracy.

In [ ]:

# Run activation experiments
import keras
from keras.models import Sequential
from keras.layers import Dense, LeakyReLU
from keras.optimizers import SGD
from sklearn.metrics import accuracy_score

def build_model(activation_name, feature_count, num_classes):
    model = Sequential()
    model.add(Dense(128, input_shape=(feature_count,)))

    if activation_name == "leakyrelu":
        model.add(LeakyReLU(alpha=0.01))
    else:
        model.add(keras.layers.Activation(activation_name))

    model.add(Dense(64))
    if activation_name == "leakyrelu":
        model.add(LeakyReLU(alpha=0.01))
    else:
        model.add(keras.layers.Activation(activation_name))

    model.add(Dense(64))
    if activation_name == "leakyrelu":
        model.add(LeakyReLU(alpha=0.01))
    else:
        model.add(keras.layers.Activation(activation_name))

    model.add(Dense(32))
    if activation_name == "leakyrelu":
        model.add(LeakyReLU(alpha=0.01))
    else:
        model.add(keras.layers.Activation(activation_name))

    model.add(Dense(num_classes, activation="softmax"))
    model.compile(loss="categorical_crossentropy", optimizer=SGD(), metrics=["accuracy"])
    return model

activations = ["relu", "leakyrelu", "tanh", "sigmoid"]
results = []

for act in activations:
    keras.utils.set_random_seed(42)  # reproducibility per run

    m = build_model(act, feature_count, num_classes)
    h = m.fit(
        X_train_nn, y_train_onehot,
        epochs=150,
        batch_size=20,
        validation_split=0.25,
        verbose=0
    )

    # Test accuracy (convert probs -> labels)
    pred_idx = np.argmax(m.predict(X_test_nn, verbose=0), axis=1)
    label_names = y_train.astype("category").cat.categories
    pred_labels = label_names[pred_idx]

    test_acc = accuracy_score(y_test, pred_labels)

    results.append({
        "activation": act,
        "best_val_accuracy": max(h.history["val_accuracy"]),
        "final_val_accuracy": h.history["val_accuracy"][-1],
        "test_accuracy": test_acc
    })

results_df = pd.DataFrame(results).sort_values("test_accuracy", ascending=False)
results_df

Activation functions were compared using the same network architecture while only changing the hidden-layer nonlinearity. Tanh performed best, achieving the highest test accuracy (0.548) compared with ReLU (0.524) and LeakyReLU (0.524), while also reaching a stable validation accuracy of 0.417. Sigmoid performed worst with test accuracy 0.214 and very low validation accuracy (0.167 → 0.125), consistent with vanishing-gradient behavior that slows or stalls learning in deeper networks. Overall, the results suggest that tanh provided a better inductive bias for this small, standardized tabular dataset, whereas sigmoid saturated too easily and failed to learn meaningful decision boundaries. ReLU and LeakyReLU remained competitive but slightly underperformed tanh in out-of-sample accuracy.

---

## 8. Explainability – SHAP Feature Importance

In this section, SHAP (SHapley Additive exPlanations) is used to interpret the model’s predictions and analyze feature importance. SHAP assigns each feature a contribution value that represents how much it influences a particular prediction. These values are based on Shapley values from cooperative game theory, which distribute the model’s prediction fairly across all contributing features.

In [ ]:
# Import necessary libraries
import shap
import matplotlib.pyplot as plt

# Transform test data using the fitted preprocessor
X_test_processed = preprocess.transform(X_test)

# Convert sparse matrix to dense if needed (required for SHAP)
if hasattr(X_test_processed, "toarray"):
    X_test_processed = X_test_processed.toarray()

# Extract feature names after preprocessing
# If unavailable, create generic feature names
try:
    feature_names = preprocess.get_feature_names_out()
except:
    feature_names = [f"feature_{i}" for i in range(X_test_processed.shape[1])]

# Initialize SHAP explainer using the trained model
explainer = shap.Explainer(best_model, X_test_processed, feature_names=feature_names)

# Compute SHAP values for the test set
shap_values = explainer(X_test_processed)

# Generate SHAP summary plot to visualize global feature importance
shap.summary_plot(shap_values, X_test_processed, feature_names=feature_names)


### Experimentation

In [ ]:

# PCA + Logistic Regression
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X_train_p = preprocess.transform(X_train)
X_test_p  = preprocess.transform(X_test)
X_train_p = X_train_p.toarray() if hasattr(X_train_p, "toarray") else X_train_p
X_test_p  = X_test_p.toarray() if hasattr(X_test_p, "toarray") else X_test_p

pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_p)
X_test_pca  = pca.transform(X_test_p)

lr = LogisticRegression(max_iter=5000)
lr.fit(X_train_pca, y_train)
pred_pca = lr.predict(X_test_pca)

print("PCA + LogReg Test Accuracy:", accuracy_score(y_test, pred_pca))
print(classification_report(y_test, pred_pca))

The PCA + Logistic Regression model achieved a test accuracy of 0.50, which is lower than the neural network model (0.62) and slightly below the regularized Random Forest (0.52). While PCA successfully reduced dimensionality and addressed multicollinearity among highly correlated development indicators, the linear decision boundaries of Logistic Regression appear insufficient to fully capture the structure of the dataset. Although the model performs well in the “Very High” category (F1 = 0.82), performance is weaker in “Very Low” (F1 = 0.29) and moderate across middle categories. This suggests that happiness classification depends on nonlinear interactions between socioeconomic variables rather than purely linear combinations of principal components. Overall, the results indicate that dimensionality reduction alone does not compensate for the need for nonlinear modeling capacity in this problem.

Deep learning models are often considered "black boxes" due to their complexity. Use SHAP to explain your model's predictions.

After applying SHAP, discuss:

- Does it provide a clear and sufficient explanation of how the model makes decisions?
- How easy or difficult is it to justify your model's predictions using this technique?

In [ ]:

import shap

# Select background (runtime control)

np.random.seed(42)

bg_idx = np.random.choice(X_train_nn.shape[0], size=min(30, X_train_nn.shape[0]), replace=False)
background = X_train_nn[bg_idx]

explain_idx = np.random.choice(X_test_nn.shape[0], size=min(20, X_test_nn.shape[0]), replace=False)
X_explain = X_test_nn[explain_idx]

# SHAP needs a predict_proba-style function for multiclass

def predict_proba(x):
    return keras_model.predict(x, verbose=0)

# Build explainer + compute SHAP values

explainer_nn = shap.KernelExplainer(predict_proba, background)
shap_values_nn = explainer_nn.shap_values(X_explain, nsamples=200)

# Feature names must match preprocessed feature count

try:
    feature_names = list(preprocess.get_feature_names_out())
except:
    feature_names = [f"feature_{i}" for i in range(X_explain.shape[1])]

# Normalize SHAP output shape to (n_samples, n_features) for one class

if isinstance(shap_values_nn, list):
    # One array per class: choose a class to visualize (e.g., last class)
    class_index = len(shap_values_nn) - 1
    sv = shap_values_nn[class_index]
else:
    sv = np.array(shap_values_nn)
    if sv.ndim == 3:
        # Handle both possible layouts
        if sv.shape[0] == X_explain.shape[0] and sv.shape[1] == X_explain.shape[1]:
            class_index = sv.shape[2] - 1
            sv = sv[:, :, class_index]
        else:
            class_index = sv.shape[0] - 1
            sv = sv[class_index, :, :]
    elif sv.ndim == 2:
        pass

# Safety check to prevent shape mismatch error
assert sv.shape == X_explain.shape, f"Mismatch: SHAP {sv.shape} vs X {X_explain.shape}"


# Global explanation plot

shap.summary_plot(sv, X_explain, feature_names=feature_names)



After applying SHAP to the neural network, the summary plot shows that the model’s predictions are primarily driven by economic and development-related variables, particularly GDP-related measures, interaction terms, freedom to make life choices, and perceptions of corruption. Higher values of these features generally push predictions toward higher happiness categories, while lower values shift predictions downward. This confirms that the neural network is relying on economically meaningful signals rather than arbitrary noise.

SHAP provides a reasonably clear global explanation of feature importance and directional impact. It is effective for verifying whether the model is using sensible variables and for identifying which features contribute most to predictions. However, explaining individual predictions remains moderately complex. Because the neural network operates on scaled and encoded features and captures nonlinear interactions, translating SHAP values into a simple, intuitive narrative requires careful interpretation. Therefore, SHAP improves transparency, but deep learning models remain more difficult to justify at a case-specific level compared to simpler or tree-based models.